# NCLT — hloc Visual Localization Results (Experiment 0.7)

Cross-season localization on the NCLT dataset:
- **Database:** Spring 2012-04-29 (every 3rd frame, 7170 images)
- **Query:** Summer 2012-08-04 (every 5th frame, 4828 images)
- **Pipeline:** SuperPoint + LightGlue + NetVLAD + COLMAP SfM + PnP
- **Camera:** Ladybug3 Cam5 (forward-facing), 808x616, 5Hz, fisheye ~120° FOV

**Spoiler:** SfM reconstruction fails to produce correct geometry — all camera positions collapse to a ~2.5m radius instead of the real ~778m route. This is a fundamental issue with the NCLT camera.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

RESULTS_DIR = Path('/workspace/nclt_pipeline/results/week0_hloc/full')
GT_DIR = Path('/workspace/data/nclt/ground_truth')

## 1. Experiment summary

In [ ]:
summary = json.load(open(RESULTS_DIR / 'summary.json'))

print('Pipeline:')
for k, v in summary['pipeline'].items():
    print(f'  {k}: {v}')

print(f'\nCamera model: {summary.get("camera_model", "SIMPLE_RADIAL")}')
print(f'DB images: {summary["n_db_images"]}')
print(f'Query images: {summary["n_query_images"]}')
print(f'Total time: {summary["total_time_s"]/3600:.1f} hours')

print(f'\n--- SfM Results ---')
sfm = summary['sfm']
print(f'Registered: {sfm["n_registered"]}/{sfm["n_total"]} ({sfm["pct_registered"]:.1f}%)')
print(f'3D points: {sfm["n_3d_points"]}')
print(f'Mean reproj error: {sfm["mean_reproj_error"]:.3f} px')
print(f'SfM Sim(3) scale: {sfm["sfm_sim3_scale"]:.2e}')
print(f'SfM ATE RMSE (Sim3): {sfm["sfm_ate_rmse"]:.4f} m')

print(f'\n--- Localization Results ---')
loc = summary['localization']
print(f'Localized: {loc["n_localized"]}/{loc["n_query"]} ({loc["pct_localized"]:.1f}%)')
print(f'Within (0.25m, 2°): {loc["pct_0.25m_2deg"]:.1f}%')
print(f'Within (0.5m, 5°): {loc["pct_0.5m_5deg"]:.1f}%')
print(f'Within (5m, 10°): {loc["pct_5.0m_10deg"]:.1f}%')
print(f'Median trans error: {loc["median_trans_error_m"]:.2f} m')
print(f'Median rot error: {loc["median_rot_error_deg"]:.2f}°')
print(f'Sim(3) scale: {loc["sim3_scale"]:.2e}')

## 2. The core problem: SfM geometry collapse

COLMAP registers most images (~80-92%) but places them all within a tiny area. The Sim(3) alignment scale of ~0 confirms this.

In [ ]:
import sys
sys.path.insert(0, '/workspace/hloc_repo')
import pycolmap

sfm_path = RESULTS_DIR / 'sfm'
if sfm_path.exists():
    model = pycolmap.Reconstruction(str(sfm_path))
    
    # Extract all camera positions
    sfm_positions = []
    for img_id, img in model.images.items():
        cfw = img.cam_from_world
        if callable(cfw): cfw = cfw()
        R = cfw.rotation.matrix()
        t = np.array(cfw.translation)
        T = np.eye(4)
        T[:3,:3] = R; T[:3,3] = t
        pos = np.linalg.inv(T)[:3, 3]
        sfm_positions.append(pos)
    
    sfm_positions = np.array(sfm_positions)
    
    # Load GT
    gt_spring = np.loadtxt(GT_DIR / 'groundtruth_2012-04-29.csv', delimiter=',')
    gt_summer = np.loadtxt(GT_DIR / 'groundtruth_2012-08-04.csv', delimiter=',')
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # SfM positions
    ax = axes[0]
    ax.scatter(sfm_positions[:, 0], sfm_positions[:, 1], s=1, alpha=0.3)
    spread = np.linalg.norm(sfm_positions.max(0) - sfm_positions.min(0))
    ax.set_title(f'SfM camera positions\n{len(sfm_positions)} cameras, spread: {spread:.1f} m')
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    
    # GT trajectory
    ax = axes[1]
    ax.plot(gt_spring[:, 1], gt_spring[:, 2], 'b-', alpha=0.5, linewidth=0.5, label='Spring GT')
    ax.plot(gt_summer[:, 1], gt_summer[:, 2], 'r-', alpha=0.5, linewidth=0.5, label='Summer GT')
    gt_spread = np.linalg.norm(gt_spring[:, 1:4].max(0) - gt_spring[:, 1:4].min(0))
    ax.set_title(f'Ground truth trajectory\nSpread: {gt_spread:.0f} m')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.set_aspect('equal')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'SfM spread: {spread:.1f}m vs GT spread: {gt_spread:.0f}m — geometry collapsed', fontsize=13)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'sfm_collapse_comparison.png', dpi=150)
    plt.show()
    
    # Camera params
    for cam_id, cam in model.cameras.items():
        print(f'\nCamera: {cam}')
else:
    print('SfM model not found')

## 3. Why does SfM fail on NCLT?

We tried two camera models:

| Camera model | Registered | SfM spread | GT spread | Scale |
|-------------|-----------|-----------|----------|-------|
| SIMPLE_RADIAL (f, cx, cy, k) | 6596/7170 (92%) | 2.5 m | 778 m | ~0 |
| OPENCV_FISHEYE (fx, fy, cx, cy, k1-k4) | 5797/7170 (81%) | ? m | 778 m | ~0 |

Both collapse. The problem is not just the camera model — it's a combination of:

1. **Extreme fisheye distortion (~120° FOV)** — even OPENCV_FISHEYE may not capture it fully
2. **Low framerate (5Hz)** — at driving speed, large baseline between consecutive frames
3. **Repetitive campus scenes** — buildings, sidewalks, trees look similar everywhere, causing false matches across distant locations
4. **No proper calibration** — we use approximate intrinsics (f=290), not factory calibration
5. **Images are not undistorted** — unlike RobotCar where images come pre-undistorted

## 4. Comparison with all NCLT methods

| Method | ATE RMSE | Notes |
|--------|----------|-------|
| LiDAR ICP | 174 m (spring) / 188 m (summer) | Only working method |
| ORB-SLAM3 Mono | N/A | Max 27.9% tracking |
| ORB-SLAM3 VIO | 378 km (!!) | 90% tracking but 310x scale error |
| DROID-SLAM | ~310 m | Trajectory doesn't follow road |
| DPVO / DPV-SLAM | ~350 m | Same issue |
| **hloc (this experiment)** | **245 m (Sim3)** | **SfM geometry collapses** |

hloc localization produces results comparable to deep SLAM methods in terms of ATE, but for a different reason:
deep methods produce wrong trajectory shapes, while hloc's SfM map itself is wrong.

In [ ]:
# Comparison bar chart
methods = ['LiDAR ICP\n(spring)', 'LiDAR ICP\n(summer)', 'hloc\n(Sim3)', 'DROID-SLAM', 'DPVO', 'ORB-SLAM3\nVIO']
ates = [174, 188, 245, 310, 350, 378000]  # ORB-SLAM3 VIO in meters
colors_bar = ['green', 'green', 'royalblue', 'orange', 'orange', 'red']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(methods[:5], ates[:5], color=colors_bar[:5], edgecolor='black', linewidth=0.5)
ax.set_ylabel('ATE RMSE (m)')
ax.set_title('NCLT Cross-Season Localization — All Methods Comparison')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, ates[:5]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f'{val}m', ha='center', fontsize=10)

# Add ORB-SLAM3 as text annotation (off the chart)
ax.annotate('ORB-SLAM3 VIO:\n378 km (!!)', xy=(4.5, 350), fontsize=9, color='red', ha='center')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'all_methods_comparison.png', dpi=150)
plt.show()

## 5. Conclusion

hloc visual localization **does not work on NCLT** due to COLMAP SfM failing to build a geometrically correct 3D map from the Ladybug3 fisheye camera.

**What could help:**
- Proper fisheye undistortion using factory calibration before feeding to COLMAP
- Using every single frame (not every 3rd) with sequential matching instead of retrieval-based
- Using a different SfM tool that natively handles equidistant fisheye models
- Using LiDAR-based 3D map instead of vision-based SfM

**Takeaway:** The NCLT Ladybug3 camera is not suitable for standard visual localization pipelines without significant preprocessing. RobotCar Seasons (with pre-undistorted images) works perfectly with the same pipeline.